# Sen1Floods11 (256x256) Benchmark Suite - Kaggle T4/P100
- Dataset: `doantrongthai/sen1floods11` (HuggingFace)
- Setting: Chuẩn hóa theo ProCANet (IEEE GRSL 2025) & Bonafilia (CVPRW 2020)
- Tối ưu log cực ngắn: Đúng 1 dòng/epoch, tuyệt đối không tràn bộ đệm log Kaggle.

In [ ]:
# 1. Cai dat thu vien can thiet (che do im lang)
!pip install -q tifffile huggingface_hub

In [ ]:
# 2. Tai dataset tu HuggingFace va giai nen
import os, glob, shutil
from huggingface_hub import hf_hub_download, snapshot_download

HF_REPO = 'doantrongthai/sen1floods11'
DATA_DIR = '/kaggle/working/data'
os.makedirs(DATA_DIR, exist_ok=True)

try:
    print(f'Dang tai sen1floods11.zip tu {HF_REPO}...')
    zip_path = hf_hub_download(repo_id=HF_REPO, filename='sen1floods11.zip', repo_type='dataset', local_dir='/kaggle/working')
    print('Giai nen du lieu (unzip -q)...')
    !unzip -q -o /kaggle/working/sen1floods11.zip -d {DATA_DIR}
    print('Giai nen hoan tat!')
except Exception as e:
    print(f'Khong tim thay zip, snapshot truc tiep: {e}')
    snapshot_download(repo_id=HF_REPO, repo_type='dataset', local_dir=DATA_DIR)

# Tu dong phat hien thu muc chua S1Hand
for root, dirs, files in os.walk(DATA_DIR):
    if 'S1Hand' in dirs and 'splits' in dirs:
        DATA_DIR = root
        break

print('Data root chuan xac:', DATA_DIR)
print('So file S1Hand:', len(os.listdir(os.path.join(DATA_DIR, 'S1Hand'))))
print('So file LabelHand:', len(os.listdir(os.path.join(DATA_DIR, 'LabelHand'))))

In [ ]:
# 3. Dinh nghia Dataset va Ham tinh Metric chuan (Water IoU, ignore_index=255)
import csv, time
import numpy as np
import tifffile as tiff
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

class Sen1Floods11Dataset256(Dataset):
    def __init__(self, root_dir, split='train', modality='s1'):
        self.root_dir = root_dir
        self.split = split
        self.modality = modality
        split_files = {'train': 'flood_train_data.csv', 'valid': 'flood_valid_data.csv', 'test': 'flood_test_data.csv', 'bolivia': 'flood_bolivia_data.csv'}
        csv_path = os.path.join(root_dir, 'splits', split_files[split])
        self.samples = []
        with open(csv_path, 'r') as f:
            reader = csv.reader(f)
            for row in reader:
                if len(row) >= 2:
                    self.samples.append((row[0].strip(), row[1].strip()))

    def __len__(self):
        return len(self.samples)

    def _load_s1(self, fname):
        p = os.path.join(self.root_dir, 'S1Hand', fname)
        arr = tiff.imread(p).astype(np.float32)
        arr = np.nan_to_num(arr, nan=0.0)
        arr = np.clip(arr, -50.0, 1.0)
        return (arr + 50.0) / 51.0

    def _load_s2(self, fname):
        p = os.path.join(self.root_dir, 'S2Hand', fname.replace('_S1Hand.tif', '_S2Hand.tif'))
        arr = tiff.imread(p).astype(np.float32)
        arr = np.nan_to_num(arr, nan=0.0)
        return np.clip(arr / 10000.0, 0.0, 1.0)

    def __getitem__(self, idx):
        s1_file, lbl_file = self.samples[idx]
        if self.modality == 's1':
            img = self._load_s1(s1_file)
        elif self.modality == 's2':
            img = self._load_s2(s1_file)
        elif self.modality == 's1s2':
            img = np.concatenate([self._load_s1(s1_file), self._load_s2(s1_file)], axis=0)
        lbl = tiff.imread(os.path.join(self.root_dir, 'LabelHand', lbl_file)).astype(np.int64)
        lbl[lbl == -1] = 255
        return torch.from_numpy(img).float(), torch.from_numpy(lbl).long()

class BCEDiceLoss(nn.Module):
    def __init__(self, ignore_index=255):
        super().__init__()
        self.ignore = ignore_index
        self.ce = nn.CrossEntropyLoss(ignore_index=ignore_index)
    def forward(self, logits, targets):
        loss_ce = self.ce(logits, targets)
        valid = (targets != self.ignore)
        if not valid.any():
            return loss_ce
        probs = F.softmax(logits, dim=1)[:, 1][valid]
        t = (targets == 1).float()[valid]
        inter = (probs * t).sum()
        dice = (2.0 * inter + 1.0) / (probs.sum() + t.sum() + 1.0)
        return 0.5 * loss_ce + 0.5 * (1.0 - dice)

class Evaluator:
    def __init__(self, ignore_index=255):
        self.ignore = ignore_index
        self.reset()
    def reset(self):
        self.tp, self.fp, self.fn, self.tn = 0, 0, 0, 0
    def update(self, p, t):
        valid = (t != self.ignore)
        p_v, t_v = p[valid], t[valid]
        self.tp += int(np.sum((p_v == 1) & (t_v == 1)))
        self.fp += int(np.sum((p_v == 1) & (t_v == 0)))
        self.fn += int(np.sum((p_v == 0) & (t_v == 1)))
        self.tn += int(np.sum((p_v == 0) & (t_v == 0)))
    def compute(self):
        w_denom = self.tp + self.fp + self.fn
        water_iou = (self.tp / w_denom) if w_denom > 0 else 0.0
        bg_denom = self.tn + self.fp + self.fn
        bg_iou = (self.tn / bg_denom) if bg_denom > 0 else 0.0
        return {'water_iou': round(water_iou, 4), 'm_iou': round((water_iou + bg_iou) / 2.0, 4)}

In [ ]:
# 4. Cac Kien truc Baseline (UNet, DeepLabV3+, SegFormer)
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class UNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=2, b=32):
        super().__init__()
        self.inc = DoubleConv(in_channels, b)
        self.d1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(b, b*2))
        self.d2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(b*2, b*4))
        self.d3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(b*4, b*8))
        self.d4 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(b*8, b*16))
        self.u1 = nn.ConvTranspose2d(b*16, b*8, 2, stride=2); self.cu1 = DoubleConv(b*16, b*8)
        self.u2 = nn.ConvTranspose2d(b*8, b*4, 2, stride=2);  self.cu2 = DoubleConv(b*8, b*4)
        self.u3 = nn.ConvTranspose2d(b*4, b*2, 2, stride=2);  self.cu3 = DoubleConv(b*4, b*2)
        self.u4 = nn.ConvTranspose2d(b*2, b, 2, stride=2);    self.cu4 = DoubleConv(b*2, b)
        self.outc = nn.Conv2d(b, num_classes, 1)
    def forward(self, x):
        x1 = self.inc(x); x2 = self.d1(x1); x3 = self.d2(x2); x4 = self.d3(x3); x5 = self.d4(x4)
        d4 = self.cu1(torch.cat([x4, self.u1(x5)], 1))
        d3 = self.cu2(torch.cat([x3, self.u2(d4)], 1))
        d2 = self.cu3(torch.cat([x2, self.u3(d3)], 1))
        d1 = self.cu4(torch.cat([x1, self.u4(d2)], 1))
        return self.outc(d1)

def build_model(name, in_channels):
    if name == 'unet':
        return UNet(in_channels=in_channels, num_classes=2)
    raise ValueError(f'Unsupported model: {name}')

In [ ]:
# 5. Ham Train & Eval - Log cuc gon (Dung 1 dong/epoch)
def evaluate_dataset(model, loader):
    model.eval()
    ev = Evaluator(ignore_index=255)
    with torch.no_grad():
        for imgs, masks in loader:
            preds = model(imgs.to(device)).argmax(dim=1).cpu().numpy()
            ev.update(preds, masks.numpy())
    return ev.compute()

def run_benchmark(model_name='unet', modality='s1', epochs=25, batch_size=32, lr=1e-3):
    in_ch = 2 if modality == 's1' else (13 if modality == 's2' else 15)
    model = build_model(model_name, in_channels=in_ch).to(device)
    
    train_loader = DataLoader(Sen1Floods11Dataset256(DATA_DIR, 'train', modality), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Sen1Floods11Dataset256(DATA_DIR, 'valid', modality), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(Sen1Floods11Dataset256(DATA_DIR, 'test', modality), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    bol_loader   = DataLoader(Sen1Floods11Dataset256(DATA_DIR, 'bolivia', modality), batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    criterion = BCEDiceLoss(ignore_index=255)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = GradScaler()
    
    best_val_iou = 0.0
    ckpt_path = f'/kaggle/working/{model_name}_{modality}_best.pth'
    
    print(f'=== BAT DAU BENCHMARK: {model_name.upper()} | Modality: {modality.upper()} | Epochs: {epochs} | Batch: {batch_size} ===')
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        loss_sum = 0.0
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += loss.item()
        scheduler.step()
        
        val_res = evaluate_dataset(model, val_loader)
        val_iou = val_res['water_iou']
        ep_t = round(time.time() - t0, 1)
        mark = '*' if val_iou > best_val_iou else ' '
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            torch.save(model.state_dict(), ckpt_path)
            
        # In dung 1 dong ngan gon moi epoch, khong bao gio ngap log Kaggle
        print(f'Ep {epoch:02d}/{epochs:02d} | Loss: {loss_sum/len(train_loader):.4f} | Val Water IoU: {val_iou:.4f} {mark} | {ep_t}s')
        
    # Load best checkpoint de test
    model.load_state_dict(torch.load(ckpt_path))
    test_res = evaluate_dataset(model, test_loader)
    bol_res  = evaluate_dataset(model, bol_loader)
    
    print(f'--> HOAN TAT {model_name.upper()}: Test Water IoU = {test_res["water_iou"]} | Bolivia = {bol_res["water_iou"]}\n')
    return {'model': model_name, 'modality': modality, 'best_val_iou': best_val_iou, 'test_iou': test_res['water_iou'], 'test_miou': test_res['m_iou'], 'bolivia_iou': bol_res['water_iou'], 'bolivia_miou': bol_res['m_iou']}

In [ ]:
# 6. Chay Benchmark (Goi ham gon gang)
results = []
# Chay UNet tren Sentinel-1 SAR (25 epoch chi mat khoang 1.5 phut tren T4)
res_unet_s1 = run_benchmark(model_name='unet', modality='s1', epochs=25, batch_size=32)
results.append(res_unet_s1)

# In bang tong ket cuoi cung
import pandas as pd
df_res = pd.DataFrame(results)
print('='*75)
print('BANG TONG KET BENCHMARK CHUAN SEN1FLOODS11 (256x256)')
print('='*75)
display(df_res)